# Lab 34: Fine-Tuning Language AI: LoRA on a Small LLM

**Duration:** 4 hours. **Level:** advanced. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
A small language model follows instructions only roughly. You can improve it for **one job** by fine-tuning it on examples of that job. Training all the numbers of a language model is too much for the Jetson. **LoRA** (low-rank adaptation) freezes the model and trains only a few small extra matrices, less than 1 % of the numbers, so it fits in about 1.6 GB and takes under a minute here. In this lab you fine-tune a small model (SmolLM2, 135 million numbers) to do one job: turn a spoken sentence about the smart building into a **JSON command** that the tools of Lab 24 can run. You measure the model **before and after** on sentences it has never seen, and you learn when fine-tuning is the right tool and when prompting or RAG is better.

**By the end of this lab you can:**
1. Explain LoRA: what is frozen, what is trained, and why it needs so little memory.
2. Build a small instruction dataset, split it so the test uses phrasings the model never saw, and format it as chat.
3. Measure a baseline (plain model and few-shot prompt) with an exact-match score.
4. Fine-tune a model with LoRA on the Jetson, save the adapter, and measure the improvement.
5. Check the limits: data size, forgetting, safety of the output, and the choice between prompting, RAG and fine-tuning.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Pictures and live video appear under the cell that makes them. Press **Ctrl + S** often.

> **Classification of this lab.** *Fine-tuning* (parameter-efficient adaptation of a pretrained model). The model is small on purpose: the language model of Labs 20 to 26 (2 billion numbers) is too large to fine-tune on the 8 GB of the Jetson.

## 2. Background

**Why not train the whole model?** Training changes every weight and needs memory for the weights, their gradients and the optimiser (about 4 times the size of the model, or more). A 2-billion-number model in 32-bit needs about 8 GB for the weights alone. The Jetson has 8 GB for everything.

**LoRA in one picture.** A weight matrix `W` of a layer is very large. LoRA leaves `W` **frozen** and adds a small correction `A x B`, where `A` and `B` are thin matrices (their width is the **rank** `r`, for example 16). Only `A` and `B` are trained.

| | Full fine-tuning | LoRA (rank 16) |
|---|---|---|
| Numbers trained | All (135 million here) | About 0.9 million (0.7 %) |
| Memory for training | High | Low (about 1.6 GB here) |
| What you save | A new copy of the model | A small **adapter** file (a few MB) |
| Risk of forgetting | Higher | Lower (the base is frozen) |

`lora_alpha` scales the correction. `target_modules` says which layers get adapters (here the attention layers `q_proj`, `k_proj`, `v_proj`, `o_proj`).

**What fine-tuning is good for.**

| Goal | Best tool |
|---|---|
| New **facts** that change (notes, prices) | **RAG** (Lab 23): update the notes, no training |
| A new **format or style** (always answer in JSON, a fixed tone) | **Fine-tuning** (this lab), or a good prompt with examples (Lab 20) |
| A **skill** on a narrow task (turn sentences into commands) | Fine-tuning, especially for a **small** model that is cheap to run |
| A bigger, cleverer model | A bigger base model; fine-tuning does not add general intelligence |

**The data.** Each example is a pair: the input sentence and the exact output wanted. The model learns by predicting the answer token by token. During training the loss is counted **only on the answer**, not on the question.

**A fair test.** Split by **phrasing**: the test sentences use words and patterns that are not in the training set. If the test were a copy of the training set, the score would only measure memory.

**Exact match.** The answer counts as right only if the JSON, once parsed, equals the wanted JSON. It is strict, and it is what a program needs.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(34)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (30 min)

### Step 1: Memory and Libraries
Training needs about 2 GB of GPU memory. The language model server of Lab 20 uses about 5 GB of the shared 8 GB. It is safest to stop it for this lab: the student who started it runs `labview.llm_stop()` (tell your team first). The cell shows the memory and installs the libraries if they are missing (internet needed).

In [ ]:
import subprocess
import sys

for package, module in (("transformers", "transformers"), ("peft", "peft"), ("accelerate", "accelerate")):
    try:
        __import__(module)
        print(f"{package:<14} is installed")
    except ImportError:
        print(f"{package:<14} installing ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import labview
print()
print(subprocess.run(["free", "-m"], capture_output=True, text=True).stdout)
print("Language model server running:", labview.llm_ready(), "(it uses about 5 GB; stop it for this lab if memory is short)")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Memory available now:
- Is the language model server running? Who must stop it:

### Step 2: Load the Small Model
`HuggingFaceTB/SmolLM2-135M-Instruct` is a small open model (Apache 2.0 licence) that was trained to follow instructions. The first time, the cell **downloads it (about 270 MB)**; on a slow network this can take several minutes. It loads the model in 32-bit numbers on the GPU.

In [ ]:
import json
import random
import time

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
SEED = 1
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(NAME)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
tokenizer.padding_side = "left"                      # for generation: the answers start at the same place


def load_base():
    torch.manual_seed(SEED)
    return AutoModelForCausalLM.from_pretrained(NAME, dtype=torch.float32).to(device)


base = load_base()
print("Model:", NAME, "| numbers:", f"{sum(p.numel() for p in base.parameters()):,}", "| device:", device)
print("GPU memory used by the model:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB" if device == "cuda" else "")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Number of weights of the model:
- GPU memory used by the model:
- Why would the 2-billion-number model of Lab 20 not fit for training (calculate: 2 billion x 4 bytes):

---
# Part B: The Dataset (40 min)

### Step 1: Make the Examples
The job: sentence in, JSON out, for the tools `set_light` and `set_fan` of Lab 24 and three rooms. Each action has 12 different phrasings. The cell uses **9 phrasings for training** and keeps **3 phrasings for the test**, so the test sentences are written in ways that the model never saw in training.

In [ ]:
ROOMS = ["kitchen", "lab", "garage"]
PHRASES = [
    ("light", True, ["turn on the light in the {r}", "switch on the {r} light", "please light up the {r}", "{r} light on please", "can you put the {r} light on",
                     "i need light in the {r}", "lights on in the {r}", "make the {r} bright", "the {r} is dark, light on",
                     "{r} lamp on", "would you switch the {r} light on", "set the {r} light to on"]),
    ("light", False, ["turn off the light in the {r}", "switch off the {r} light", "kill the lights in the {r}", "{r} light off please", "can you put the {r} light out",
                      "i do not need light in the {r}", "lights off in the {r}", "make the {r} dark", "the {r} is empty, light off",
                      "{r} lamp off", "would you switch the {r} light off", "set the {r} light to off"]),
    ("fan", True, ["start the fan in the {r}", "switch on the {r} fan", "{r} is hot, fan on", "please run the {r} fan", "fan on in the {r}",
                   "i am hot in the {r}", "cool the {r} down", "turn the {r} fan on", "the {r} needs air, fan on",
                   "{r} fan on please", "would you start the {r} fan", "set the {r} fan to on"]),
    ("fan", False, ["stop the fan in the {r}", "switch off the {r} fan", "{r} fan off please", "you can shut the {r} fan", "fan off in the {r}",
                    "i am cold in the {r}", "stop cooling the {r}", "turn the {r} fan off", "the {r} is cool, fan off",
                    "{r} fan off now", "would you stop the {r} fan", "set the {r} fan to off"]),
]


def make(indices):
    rows = []
    for device_name, on, phrases in PHRASES:
        for i in indices:
            for room in ROOMS:
                rows.append((phrases[i].format(r=room), json.dumps({"tool": "set_" + device_name, "room": room, "on": on})))
    return rows


train_data = make(range(9))
test_data = make(range(9, 12))
random.Random(SEED).shuffle(train_data)
print("training examples:", len(train_data), "| test examples:", len(test_data))
for sentence, answer in train_data[:3] + test_data[:2]:
    print(f"  {sentence:<38} -> {answer}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The size of the training set and of the test set:
- Why are the test phrasings different from the training phrasings:
- Write one more phrasing for `light on` that is not in the lists:

### Step 2: The Chat Format
The model was trained with a **chat template**: the question is put in a special wrapper so that the model knows where its answer starts. The function `prompt` builds the question; the training text is the prompt followed by the answer. Look at the exact text.

In [ ]:
def prompt(sentence, shots=()):
    messages = []
    for question, answer in shots:                                   # examples before the real question (few-shot, Part C)
        messages += [{"role": "user", "content": "Turn this into a JSON command: " + question}, {"role": "assistant", "content": answer}]
    messages.append({"role": "user", "content": "Turn this into a JSON command: " + sentence})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


example_sentence, example_answer = train_data[0]
print(prompt(example_sentence) + example_answer + tokenizer.eos_token)
print("\nTokens in this text:", len(tokenizer(prompt(example_sentence) + example_answer)["input_ids"]))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The special words around the question and the answer:
- How many tokens one example has:

---
# Part C: The Baseline (40 min)

### Step 1: The Score Function
`generate` asks the model for the answers to many sentences at once (a batch), without randomness (`do_sample=False`). `exact_match` parses the answer as JSON and compares it with the wanted JSON. The cell tests the **plain model** on the test sentences.

In [ ]:
def generate(model, sentences, shots=(), batch=12, max_new_tokens=32):
    answers = []
    model.eval()
    for start in range(0, len(sentences), batch):
        texts = [prompt(s, shots) for s in sentences[start:start + batch]]
        encoded = tokenizer(texts, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            out = model.generate(**encoded, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        answers += [tokenizer.decode(row[encoded["input_ids"].shape[1]:], skip_special_tokens=True).strip() for row in out]
    return answers


def exact_match(model, data, shots=(), show=0):
    started = time.time()
    answers = generate(model, [s for s, _ in data], shots)
    right = 0
    for (sentence, wanted), got in zip(data, answers):
        try:
            ok = json.loads(got) == json.loads(wanted)
        except json.JSONDecodeError:
            ok = False
        right += ok
        if show and not ok and show > 0:
            show -= 1
            print(f"  WRONG: {sentence!r}\n     got   {got[:90]!r}\n     wanted {wanted}")
    print(f"exact match {right} of {len(data)} = {right / len(data) * 100:.0f} %   ({time.time() - started:.0f} s)")
    return right / len(data)


print("Plain model:")
score_plain = exact_match(base, test_data, show=2)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The exact match of the plain model:
- What does the plain model write instead of the JSON:

### Step 2: A Few Examples in the Prompt (Few-Shot)
In Lab 20 you gave the model examples in the prompt. Give it 4 examples from the training set before each question, and measure again. This costs nothing to train, but makes every prompt longer.

In [ ]:
shots = train_data[:4]
print("Plain model with 4 examples in the prompt:")
score_few = exact_match(base, test_data, shots=shots, show=2)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The exact match with 4 examples:
- Is few-shot prompting enough for this small model? What does it cost on every call:

---
# Part D: Fine-Tune with LoRA (60 min)

### Step 1: Add the Adapters
`get_peft_model` freezes the model and adds LoRA matrices to the attention layers. The cell prints how many numbers will be trained.

In [ ]:
from peft import LoraConfig, get_peft_model

RANK = 16


def with_lora(model, rank=RANK):
    config = LoraConfig(r=rank, lora_alpha=2 * rank, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], lora_dropout=0.05, task_type="CAUSAL_LM")
    return get_peft_model(model, config)


model = with_lora(base)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trained: {trainable:,} of {total:,} numbers = {trainable / total * 100:.2f} %")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Trained numbers and the share:
- Which layers got adapters:

### Step 2: The Training Loop
Each step takes a batch of 8 examples. The text of each example is prompt + answer. The **labels** copy the text but hide the prompt part (value -100), so that the loss is calculated only on the answer. Four epochs of 108 examples are 56 steps. The cell prints the loss: it should fall from above 1 to almost 0.

In [ ]:
def make_batch(items):
    texts = [prompt(s) + a + tokenizer.eos_token for s, a in items]
    tokenizer.padding_side = "right"                                  # for training the padding goes to the right
    encoded = tokenizer(texts, return_tensors="pt", padding=True).to(device)
    tokenizer.padding_side = "left"
    labels = encoded["input_ids"].clone()
    labels[encoded["attention_mask"] == 0] = -100                     # no loss on the padding
    for i, (sentence, _) in enumerate(items):
        labels[i, :len(tokenizer(prompt(sentence))["input_ids"])] = -100      # no loss on the question
    return encoded, labels


def fine_tune(model, data, epochs=4, lr=3e-4, batch=8, show_every=10):
    optimiser = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    rng = random.Random(SEED)
    model.train()
    step, history, started = 0, [], time.time()
    for epoch in range(epochs):
        items = list(data)
        rng.shuffle(items)
        for start in range(0, len(items), batch):
            encoded, labels = make_batch(items[start:start + batch])
            loss = model(**encoded, labels=labels).loss
            loss.backward()
            optimiser.step()
            optimiser.zero_grad()
            step += 1
            history.append(loss.item())
            if show_every and step % show_every == 0:
                print(f"step {step:>3}  loss {loss.item():.3f}")
    print(f"{step} steps in {time.time() - started:.0f} s | highest GPU memory {torch.cuda.max_memory_allocated() / 1e9:.2f} GB" if device == "cuda" else f"{step} steps")
    return history


history = fine_tune(model, train_data)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The loss at the first printed step and at the last:
- Time of the training and the highest memory:
- Compare with training all 135 million numbers: what changes in the memory and why:

### Step 3: Save the Adapter
Only the LoRA matrices are saved. The file is small. To use it later you load the base model and the adapter.

In [ ]:
import os

model.save_pretrained("adapter_commands")
size = sum(os.path.getsize(os.path.join("adapter_commands", f)) for f in os.listdir("adapter_commands"))
print("Files:", sorted(os.listdir("adapter_commands")))
print(f"Adapter size: {size / 1e6:.1f} MB (the base model is about 540 MB in 32-bit)")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The size of the adapter:
- Why can many adapters (one per job) share one base model:

---
# Part E: Measure the Improvement (40 min)

### Step 1: Before and After
The same test sentences, the same strict score. Compare with the plain model and with few-shot prompting.

In [ ]:
print("Fine-tuned model (LoRA):")
score_tuned = exact_match(model, test_data, show=3)
print()
print(f"plain {score_plain * 100:.0f} %   few-shot {score_few * 100:.0f} %   LoRA {score_tuned * 100:.0f} %")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The three scores:
- If some answers are wrong, what kind of mistake is it (the room, on or off, or the tool):
- Was the test set seen during training? Explain why this score is honest:

### Step 2: Sentences Nobody Planned
Try your own sentences, also strange ones. The model has learned the **format**, but the understanding comes from the base model.

In [ ]:
own = ["it is really hot in the garage", "kitchen light off", "could you turn the lab fan on for me", "play some music", "switch on the light in the bedroom"]
for sentence, answer in zip(own, generate(model, own)):
    print(f"{sentence!r:<46} -> {answer}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which of your sentences gave a good command:
- What happened with `play some music` and with the bedroom, which are not in the data:
- Is a JSON answer for an impossible request a problem? What should the program do (see Part F):

---
# Part F: Use It Safely (30 min)

### Step 1: From Sentence to Action
The output of a model is **text**. Before a program runs it, it must be parsed and checked, exactly as in Lab 24. The cell connects the fine-tuned model to the tools of the simulated building and refuses everything that is not a valid command.

In [ ]:
import sensor_tools as building

ALLOWED = {"set_light", "set_fan"}


def run_sentence(sentence):
    text = generate(model, [sentence])[0]
    try:
        command = json.loads(text)
        tool, room, on = command["tool"], command["room"], command["on"]
        assert tool in ALLOWED and isinstance(on, bool) and room in building.STATE, "not an allowed command"
    except Exception as error:
        return f"REFUSED ({type(error).__name__}): {text[:60]!r}"
    return building.FUNCTIONS[tool](room, on)


building.reset()
for sentence in ["switch on the garage light", "kitchen fan on please", "play some music", "switch on the light in the bedroom"]:
    print(f"{sentence!r:<42} -> {run_sentence(sentence)}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which sentences were run and which were refused:
- Why does the program check the command even after fine-tuning:
- What would you add before this controls a real door (Lab 26: approval):

---
# Part G: Limits and Experiments (45 min)

### Step 1: How Much Data Do You Need?
Train a new adapter with 9, 27 and 108 examples and measure each on the test set. Each run loads the base model again (a few seconds) and trains in a few seconds to under half a minute.

In [ ]:
sizes = (9, 27, 108)
curve = {}
for size in sizes:
    fresh = with_lora(load_base())
    fine_tune(fresh, train_data[:size], epochs=4 if size > 27 else 8, show_every=0)
    curve[size] = exact_match(fresh, test_data)
    del fresh
    torch.cuda.empty_cache()
print({k: f"{v * 100:.0f} %" for k, v in curve.items()})

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The scores for 9, 27 and 108 examples:
- How many examples are enough for this job:
- What does this tell you about the work of collecting data:

### Step 2: Did It Forget Something?
Fine-tuning on one narrow job can damage other skills. Ask both models a general question and compare. The LoRA base is frozen, so the damage is usually small, but the answer style changes.

In [ ]:
question = "What is the capital of France?"
for name, m in (("plain model", base), ("fine-tuned", model)):
    encoded = tokenizer(tokenizer.apply_chat_template([{"role": "user", "content": question}], tokenize=False, add_generation_prompt=True), return_tensors="pt").to(device)
    with torch.no_grad():
        out = (m.generate(**encoded, max_new_tokens=24, do_sample=False, pad_token_id=tokenizer.pad_token_id) if name == "plain model" else m.generate(**encoded, max_new_tokens=24, do_sample=False, pad_token_id=tokenizer.pad_token_id))
    print(f"{name:<12}: {tokenizer.decode(out[0][encoded['input_ids'].shape[1]:], skip_special_tokens=True).strip()!r}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The two answers:
- Did the fine-tuned model keep its general knowledge? Is it now only a command writer:

### Step 3: Choose the Tool
Fill in the table (double-click to edit): for each job, write whether you would use **prompting**, **RAG** or **fine-tuning**, and why.

| Job | Your choice | Why |
|---|---|---|
| Answer questions about the class notes that change every week | | |
| Always answer in a strict JSON format for a small model | | |
| Make the helper speak like a pirate for one afternoon | | |
| Understand the names of the 500 parts in a factory catalogue | | |

---
# Part H: Make It Your Own and Present (30 min)
Choose **one**:

- Add a third device (for example `set_door`... only with the approval of Lab 26), or a fourth room, and change the dataset and the test.
- Change the rank to 4 and 64 and compare the trained numbers, the training time and the score.
- Add **negative examples**: for sentences that are not commands (`play some music`), the wanted answer is `{"tool": "none"}`. Does the model now refuse them?
- Fine-tune on Malay phrasings (*buka lampu dapur*) and test.

Then give a **4-minute demonstration**: the dataset and the split, the baseline, the LoRA numbers, the loss, the before-and-after score, the safe runner and the data-size curve.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- What you saw:
- When you would fine-tune and when you would not:

> **Where next.** This is the end of Part 3. You can adapt a vision model (Lab 32), a speech model (Lab 33) and a language model (this lab). In Part 4 you go one level deeper and build the models yourself.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The model is loaded and the numbers and the memory are reported | |
| 2 | The dataset is made with a test split by phrasing, and the chat format is shown | |
| 3 | The baseline (plain and few-shot) is measured with the exact-match score | |
| 4 | The LoRA numbers are counted and the training loss falls | |
| 5 | The adapter is saved and its size is reported | |
| 6 | The before-and-after scores are compared, with the errors read | |
| 7 | The safe runner refuses an invalid or impossible command | |
| 8 | The data-size experiment and the forgetting test are done | |
| 9 | One improvement is done and the 4-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers**

- 1. What is frozen in LoRA and what is trained? Why does it need little memory?:
- 2. What do `r` and `target_modules` mean?:
- 3. Why is the loss counted only on the answer?:
- 4. Why must the test phrasings be different from the training phrasings?:
- 5. What is exact match? Why is it a good score for a program that needs JSON?:
- 6. Few-shot prompting or fine-tuning: which has no training cost and which has a longer prompt on every call?:
- 7. Why should the program check the output of a fine-tuned model?:
- 8. When would you choose RAG and not fine-tuning?:
- 9. Why is the adapter much smaller than the model, and what is the benefit?:
- 10. Name one risk of fine-tuning on a very small or biased dataset.:

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| The download of the model is very slow or stops | The network is slow | Run the cell again (it continues). Ask your teacher to download `HuggingFaceTB/SmolLM2-135M-Instruct` once for the whole class. |
| `CUDA out of memory` | The language model server and other notebooks use the memory | Stop the server (`labview.llm_stop()`), shut down other kernels, restart this kernel |
| `ModuleNotFoundError: peft` | Part A Step 1 was not run, or there is no internet | Run it again |
| The exact match after training is 0 % | The labels were not set (the model learns nothing), or the learning rate is far too low | Check `make_batch`; use `lr = 3e-4` and 4 epochs |
| The loss goes to 0 but the test score is low | The training data is too small or too similar | Add more phrasings; look at the wrong answers |
| `RuntimeError: ... device` | Part of the model is on the CPU | Run Part A Step 2 again and move the model with `.to(device)` |
| The generation is slow | The batch is too small, or the CPU is used | Use `batch = 12`; check `device` |
| The scores change when I run again | Another seed or another order of cells | Run from the top; `SEED = 1` |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(34)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- Hu et al., LoRA: Low-Rank Adaptation of Large Language Models: https://arxiv.org/abs/2106.09685
- Hugging Face PEFT: https://huggingface.co/docs/peft
- SmolLM2: https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct